## ***Lets use GeoPanda & MAP Location for the event of Flick Button Press***

In [1]:
# pip install geopy folium     where geopy → convert GPS to address  ,  Folium → create interactive map

# Importing the required libraries
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable, GeocoderServiceError

## ***Loading Data which has 5 min time gap for each event. Here we assume that within 5 min short time window new hazard cannot happen. Because truck is not a high-speed vehicle which cannot travel long distances in a few minutes. Further when a hazard occurs==>> the truck will typically slow down or stop to handle the situation which takes additional time.Therefore, multiple records within 5 minutes are likely related to the same hazard event rather than separate ones.***

In [2]:
# load the data
csv_path_1 = r"C:\SUBASH\LAPIN_AMK\SPRING2026\PROJECT\EDA\Hazard_Situation_Analysis_and_Prediction\DATA\button_press_count_per_5_min_geo_map.csv"
df_geo_map = pd.read_csv(csv_path_1)

df_geo_map.head()

,ts_min,lat,lon,rssi,ts_5min,street,city,country,address
0,2025-11-10 01:50:00,60.493152,22.288016,-60,2025-11-10 01:50:00,Telekatu,Turku,Suomi / Finland,"1, Telekatu, Urusvuoren teollisuusalue, Leaf, ..."
1,2025-11-10 01:50:00,60.493151,22.288017,-63,2025-11-10 01:50:00,Telekatu,Turku,Suomi / Finland,"1, Telekatu, Urusvuoren teollisuusalue, Leaf, ..."
2,2025-11-10 01:50:00,60.493150,22.288017,-76,2025-11-10 01:50:00,Telekatu,Turku,Suomi / Finland,"1, Telekatu, Urusvuoren teollisuusalue, Leaf, ..."
3,2025-11-10 01:50:00,60.493148,22.288014,-55,2025-11-10 01:50:00,Telekatu,Turku,Suomi / Finland,"1, Telekatu, Urusvuoren teollisuusalue, Leaf, ..."
4,2025-11-10 01:50:00,60.493146,22.288012,-54,2025-11-10 01:50:00,Telekatu,Turku,Suomi / Finland,"1, Telekatu, Urusvuoren teollisuusalue, Leaf, ..."


## ***In this 5 min take MAX RSSI and Get Time, Lat, Lon, Street & City based on MAX RSSI***

In [3]:
# Find row with maximum RSSI in each 5-minute interval
idx = df_geo_map.groupby("ts_5min")["rssi"].idxmax()

# Create new dataframe using those rows
df_5min = df_geo_map.loc[idx, ["ts_5min", "lat", "lon", "rssi", "street", "city" ]].copy()
# Rename columns
df_5min = df_5min.rename(columns={
    "ts_5min": "datetime",
    "rssi": "max_rssi"
})

# Add press count per 5-minute interval
df_5min["press_count"] = df_geo_map.groupby("ts_5min").size().values

# Reset index
df_5min = df_5min.reset_index(drop=True)

# View results
print(df_5min.head())
print(df_5min.shape)

              datetime        lat        lon  max_rssi          street  \
0  2025-11-10 01:50:00  60.493146  22.288012       -54        Telekatu   
1  2025-11-10 04:05:00  60.378961  23.104875       -52  Meriniitynkatu   
2  2025-11-10 04:25:00  60.204217  23.112953       -51      Perniöntie   
3  2025-11-10 05:40:00  60.009216  23.545273       -52      Hangövägen   
4  2025-11-10 09:50:00  59.819318  22.958064       -53  Graniittikaari   

             city  press_count  
0           Turku           11  
1  Salon suuralue           39  
2            Salo           37  
3        Raseborg           37  
4           Hanko           16  
(23, 7)


## ***We have 23 button press***

In [4]:
df_5min.reset_index(drop=True, inplace=True)
df_5min.head(30)

,datetime,lat,lon,max_rssi,street,city,press_count
0,2025-11-10 01:50:00,60.493146,22.288012,-54,Telekatu,Turku,11
1,2025-11-10 04:05:00,60.378961,23.104875,-52,Meriniitynkatu,Salon suuralue,39
2,2025-11-10 04:25:00,60.204217,23.112953,-51,Perniöntie,Salo,37
3,2025-11-10 05:40:00,60.009216,23.545273,-52,Hangövägen,Raseborg,37
4,2025-11-10 09:50:00,59.819318,22.958064,-53,Graniittikaari,Hanko,16
5,2025-11-10 09:55:00,59.819312,22.958067,-54,Graniittikaari,Hanko,14
6,2025-11-10 18:05:00,60.385498,23.161347,-55,Ahmankatu,Salon suuralue,29
7,2025-11-27 00:05:00,60.235293,23.586860,-51,Kiskontie,Salo,122
8,2025-11-27 00:10:00,60.250072,23.479376,-53,Kiskontie,Salo,25
9,2025-11-27 13:05:00,60.487693,22.173565,-57,Turun kehätie,Raisio,88


## ***Lets calculate real geo distance from each button press event to next***

In [5]:
import numpy as np

# make sure sorted by datetime
df_5min = df_5min.sort_values("datetime").reset_index(drop=True)

# previous row lat/lon
df_5min["prev_lat"] = df_5min["lat"].shift(1)
df_5min["prev_lon"] = df_5min["lon"].shift(1)

# haversine function in km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# distance from previous row
df_5min["distance_km"] = haversine_km(
    df_5min["prev_lat"],
    df_5min["prev_lon"],
    df_5min["lat"],
    df_5min["lon"]
)

# optional: remove helper columns
df_5min = df_5min.drop(columns=["prev_lat", "prev_lon"])

df_5min.head(30)

,datetime,lat,lon,max_rssi,street,city,press_count,distance_km
0,2025-11-10 01:50:00,60.493146,22.288012,-54,Telekatu,Turku,11,NaN
1,2025-11-10 04:05:00,60.378961,23.104875,-52,Meriniitynkatu,Salon suuralue,39,46.579056
2,2025-11-10 04:25:00,60.204217,23.112953,-51,Perniöntie,Salo,37,19.435745
3,2025-11-10 05:40:00,60.009216,23.545273,-52,Hangövägen,Raseborg,37,32.313324
4,2025-11-10 09:50:00,59.819318,22.958064,-53,Graniittikaari,Hanko,16,38.951681
5,2025-11-10 09:55:00,59.819312,22.958067,-54,Graniittikaari,Hanko,14,0.000688
6,2025-11-10 18:05:00,60.385498,23.161347,-55,Ahmankatu,Salon suuralue,29,63.957146
7,2025-11-27 00:05:00,60.235293,23.586860,-51,Kiskontie,Salo,122,28.777723
8,2025-11-27 00:10:00,60.250072,23.479376,-53,Kiskontie,Salo,25,6.155367
9,2025-11-27 13:05:00,60.487693,22.173565,-57,Turun kehätie,Raisio,88,76.495149


## ***If the distance from current button press to the next is less than a 1 KM then merge it to current one. We do this to further remove GPS noises***

## ***Here we have only 3 for less than 1 Km***

In [6]:
df_5min["merge_with_previous"] = df_5min["distance_km"] < 1
df_5min[df_5min["merge_with_previous"]]


,datetime,lat,lon,max_rssi,street,city,press_count,distance_km,merge_with_previous
5,2025-11-10 09:55:00,59.819312,22.958067,-54,Graniittikaari,Hanko,14,0.000688,True
11,2025-11-27 14:05:00,60.431510,22.995903,-57,Turku-Helsinki moottoritie,Salo,21,0.942552,True
22,2026-01-09 22:30:00,60.325976,22.943136,-51,Kokkilantie,Salo,11,0.301776,True


## ***Lets merge less than 1Km to previous count***

In [7]:
df_5min["group"] = (df_5min["distance_km"] >= 1).cumsum()

In [8]:
df_merged = (
    df_5min
    .groupby("group")
    .agg(
        datetime=("datetime", "first"),
        lat=("lat", "first"),
        lon=("lon", "first"),
        max_rssi=("max_rssi", "max"),
        street=("street", "first"),
        city=("city", "first"),
        press_count=("press_count", "sum"),
        distance_km=("distance_km", "first")   # keep distance
    )
    .reset_index(drop=True)
)

df_geo_map = df_merged
df_geo_map["distance_km"] = df_geo_map["distance_km"].fillna(0)
df_geo_map.reset_index(drop=True, inplace=True)
df_geo_map.head(30)

,datetime,lat,lon,max_rssi,street,city,press_count,distance_km
0,2025-11-10 01:50:00,60.493146,22.288012,-54,Telekatu,Turku,11,0.000000
1,2025-11-10 04:05:00,60.378961,23.104875,-52,Meriniitynkatu,Salon suuralue,39,46.579056
2,2025-11-10 04:25:00,60.204217,23.112953,-51,Perniöntie,Salo,37,19.435745
3,2025-11-10 05:40:00,60.009216,23.545273,-52,Hangövägen,Raseborg,37,32.313324
4,2025-11-10 09:50:00,59.819318,22.958064,-53,Graniittikaari,Hanko,30,38.951681
5,2025-11-10 18:05:00,60.385498,23.161347,-55,Ahmankatu,Salon suuralue,29,63.957146
6,2025-11-27 00:05:00,60.235293,23.586860,-51,Kiskontie,Salo,122,28.777723
7,2025-11-27 00:10:00,60.250072,23.479376,-53,Kiskontie,Salo,25,6.155367
8,2025-11-27 13:05:00,60.487693,22.173565,-57,Turun kehätie,Raisio,88,76.495149
9,2025-11-27 14:00:00,60.434147,22.979577,-57,Turku-Helsinki moottoritie,Salo,28,44.585438


## ***Lets Create Folium MAP***

In [9]:
import folium
from folium.plugins import HeatMap

df_map = df_geo_map.sort_values("datetime").reset_index(drop=True)

center_lat = df_map["lat"].mean()
center_lon = df_map["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# heatmap
heat_data = df_map[["lat", "lon", "press_count"]].values.tolist()

HeatMap(
    heat_data,
    radius=25,
    blur=18,
    min_opacity=0.4,
    gradient={
        0.2: "green",
        0.5: "yellow",
        0.8: "orange",
        1.0: "red"
    }
).add_to(m)

for _, row in df_map.iterrows():

    # circle marker
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="black",
        fill=True,
        fill_opacity=0.8,
        popup=f"Datetime: {row['datetime']}<br>Press count: {row['press_count']}"
    ).add_to(m)

    # label above marker
    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size:11px;
                font-weight:bold;
                color:black;
                background-color:white;
                border-radius:4px;
                padding:1px 4px;
                transform: translate(-50%, -120%);
                ">
                {row['press_count']}
            </div>
            """
        )
    ).add_to(m)

m

In [10]:
m.save("button_press_heatmap.html")

## ***Lets focus on press_count we can see on above MAP***

In [11]:
df_press = (
    df_geo_map[["datetime", "street", "city", "press_count"]]
    .copy()
    .sort_values(by="press_count", ascending=False)
    .reset_index(drop=True)
)

df_press.head(50)

,datetime,street,city,press_count
0,2025-11-27 15:00:00,Kiskontie,Salo,131
1,2025-11-27 00:05:00,Kiskontie,Salo,122
2,2025-11-27 13:05:00,Turun kehätie,Raisio,88
3,2025-11-27 14:55:00,Kiskontie,Salo,71
4,2026-01-09 03:40:00,Meriniityntie,Salo,56
5,2026-01-09 16:30:00,Iso-Iivarintie,Paimio,48
6,2025-11-10 04:05:00,Meriniitynkatu,Salon suuralue,39
7,2025-11-10 04:25:00,Perniöntie,Salo,37
8,2025-11-10 05:40:00,Hangövägen,Raseborg,37
9,2025-11-27 15:10:00,Salovägen,Raseborg,36
